## Optional notebook— Text and token IDs (~5 minutes)

Editable learner copy with Reset to this approved baseline. No training, GPU, model download or checkpoint required. Browser notebook dependency: Pyodide's existing `pyodide.http.pyfetch`; remaining operations use Python built-ins.

### Cell 1 — Markdown

“Can we turn text into numbers without losing it? We will begin with Hello, then use the same idea on Tiny Shakespeare. Before running each cell, predict what will happen. Run it, change one thing, and explain the result. This notebook explores representation; it does not train a neural network.”

Predict: `Hello` contains five positions. How many distinct characters does it contain? Four, because `l` repeats. We will check that distinction in code.

### Cell 2 — Code: predict and test the toy mapping

Before running: predict the encoded list and whether decoding will return exactly `Hello`. Use the lookup card from Page 2.

In [ ]:
toy_to_id = {"H": 0, "e": 1, "l": 2, "o": 3, " ": 4}
toy_to_character = {index: char for char, index in toy_to_id.items()}
example = "Hello"
example_ids = [toy_to_id[char] for char in example]
restored = "".join(toy_to_character[index] for index in example_ids)
print("Text:", repr(example))
print("IDs:", example_ids)
print("Restored:", repr(restored))
assert restored == example


Expected: `[0, 1, 2, 2, 3]` and the restored string `Hello`. Both `l` positions use `2`. Change `example` to `Hello Hello`, predict where `4` appears, and run again. It appears at the space; the same character IDs are reused.

### Cell 3 — Code: build the real dataset vocabulary

The toy mapping was chosen for readability. Now let the dataset determine the characters. Predict: must its ID for `l` equal `2`? No—the mapping can differ. This cell downloads the text, so it needs a network connection.

In [ ]:
from pyodide.http import pyfetch

url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
response = await pyfetch(url)
if not response.ok:
    raise RuntimeError(f"Dataset download failed: HTTP {response.status}. Retry this cell.")
text = await response.string()
print("Characters:", len(text))
print(repr(text[:120]))
characters = sorted(set(text))
character_to_id = dict(zip(characters, range(len(characters))))
id_to_character = dict(enumerate(characters))
print("Vocabulary size:", len(characters))
print([(repr(c), character_to_id[c]) for c in characters[:12]])


Expected: nonempty text, a vocabulary size, and distinct IDs including whitespace. `set` collects distinct characters; sorting gives a consistent order; the dictionaries map in both directions. Exact dataset values are not claimed as executed results. Before publication, pin/cache the dataset and record its hash; this URL currently follows the upstream preparation script's mutable dataset URL.

### Cell 4 — Code: inspect several sequences

Predict: when a character occurs in two different snippets, should its ID change? No. The dictionary stays fixed; only the sequence being encoded changes. Run this cell and use the printed character/ID pairs to check your prediction.

In [ ]:
def encode(value):
    unknown = set(value) - set(character_to_id)
    if unknown:
        raise ValueError(f"Characters absent from this vocabulary: {sorted(unknown)!r}")
    return [character_to_id[c] for c in value]

def decode(ids):
    return "".join(id_to_character[i] for i in ids)

for start in (0, 40, 80):
    snippet = text[start:start + 16]
    ids = encode(snippet)
    print("Text:", repr(snippet))
    print("IDs: ", ids)
    print("Pairs:", list(zip(map(repr, snippet), ids)))
    assert decode(ids) == snippet


Expected: three text/ID comparisons and successful round trips. These slices are short only to make the output readable.

Then change the start positions or preview length. Explain why different snippets give different lists, yet the same character keeps the same ID. This is the difference between changing the input and changing the representation scheme.

### Cell 5 — Markdown: try it

1. Return to the toy example. Predict what happens with `Hello!`, then try it: the toy dictionary has no `!`, so its direct lookup raises `KeyError`. An ID cannot be silently invented while keeping the trained vocabulary unchanged. Restore the valid example afterward.
2. Choose a character actually present in the downloaded text. Find its ID in two snippets. Explain why it stays the same even when the surrounding text differs.
3. Has any cell learned to predict the next character? No. We constructed and tested reversible lookups. Prediction requires a model; learning requires parameter updates.

**Finish by explaining:** “Encoding changes the representation of text; it does not learn which token should come next.” If the round-trip assertion fails after an edit, check that encoding and decoding use matching dictionaries. Reset returns your copy to the baseline; your experiments must not change another learner's notebook.